# Assignment 5: End-to-End NLP Pipeline (NLTK)

**Corpus choice:** Option A — Gutenberg corpus, file `carroll-alice.txt`.


In [1]:
!pip install nltk gensim scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 35.5 MB/s eta 0:00:00


In [2]:
import nltk
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')   # required on some Colab/NLTK versions
nltk.download('stopwords')


[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Part A — Text Preprocessing (50%)


In [3]:
from nltk.corpus import gutenberg
from nltk.tokenize import word_tokenize

fileid = 'carroll-alice.txt'
raw_text = gutenberg.raw(fileid)

print("Chosen corpus:", fileid)
print("Total characters:", len(raw_text))

tokens_before = word_tokenize(raw_text)
print("Total tokens BEFORE preprocessing:", len(tokens_before))


Chosen corpus: carroll-alice.txt
Total characters: 144395
Total tokens BEFORE preprocessing: 33535


In [4]:
import re
from collections import Counter
from nltk.tokenize import word_tokenize

def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if re.search(r"[a-z0-9]", t)]
    return tokens

tokens = preprocess(raw_text)

print("Total tokens AFTER preprocessing:", len(tokens))
vocab = set(tokens)
print("Vocabulary size:", len(vocab))

freq = Counter(tokens)
print("Top 20 most frequent tokens:")
for w, c in freq.most_common(20):
    print(w, c)


Total tokens AFTER preprocessing: 27195
Vocabulary size: 2794
Top 20 most frequent tokens:
the 1616
and 810
to 720
a 631
she 545
i 542
it 540
of 499
said 462
alice 397
was 367
in 359
you 359
that 284
as 256
her 248
n't 217
at 209
's 201
on 191


Preprocessing choices strongly affect downstream NLP tasks. Lowercasing reduces sparsity by merging word variants like “Alice” and “alice”, which helps vectorization and embeddings learn more stable patterns. Removing punctuation simplifies the vocabulary, but it may remove useful signals for language modeling such as sentence boundaries and dialogue style. Keeping stopwords can be beneficial for embeddings because they provide syntactic context, while removing them can make TF-IDF focus more on content words. If stemming or lemmatization is applied, it can improve generalization by merging word forms, but it may reduce interpretability or blur meaning. Overall, lighter preprocessing tends to work well for language modeling and embeddings, while stronger normalization can be helpful for classic document representations.

## Part B — Text Representation (25%)


### B1. Document creation strategy (justification)
I split the corpus into fixed-size chunks of **800 tokens per document**. The Gutenberg text does not have reliable, consistent chapter boundaries in the raw file, so chunking creates multiple comparable “documents” of similar length for vectorization and similarity analysis.


In [5]:
def chunk_tokens(tokens, chunk_size=800):
    docs = []
    for i in range(0, len(tokens), chunk_size):
        chunk = tokens[i:i+chunk_size]
        docs.append(" ".join(chunk))
    return docs

documents = chunk_tokens(tokens, chunk_size=800)
print("Number of documents:", len(documents))


Number of documents: 34


In [6]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vec = CountVectorizer()
X_bow = count_vec.fit_transform(documents)
print("BoW shape (num_docs x vocab):", X_bow.shape)

tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(documents)
print("TF-IDF shape (num_docs x vocab):", X_tfidf.shape)


BoW shape (num_docs x vocab): (34, 2547)
TF-IDF shape (num_docs x vocab): (34, 2547)


### B2. Top TF‑IDF terms (interpretation)
Below are the top TF‑IDF terms for **two different documents**. TF‑IDF highlights words that are frequent in one document but relatively rare across other documents, so the top terms often reflect the local topic/scene.


In [7]:
import numpy as np

feature_names = np.array(tfidf_vec.get_feature_names_out())

def top_tfidf_terms(doc_index, top_n=15):
    row = X_tfidf[doc_index].toarray().flatten()
    top_ids = row.argsort()[::-1][:top_n]
    return list(zip(feature_names[top_ids], row[top_ids]))

for idx in [0, 1]:
    print("\nDoc", idx, "Top TF-IDF terms:")
    for term, score in top_tfidf_terms(idx, 15):
        print(term, round(score, 4))



Doc 0 Top TF-IDF terms:
the 0.3548
to 0.3153
she 0.2562
it 0.2464
was 0.1971
and 0.1774
of 0.1774
down 0.1718
her 0.1356
rabbit 0.123
alice 0.1084
think 0.1054
pictures 0.1022
as 0.0985
very 0.0939

Doc 1 Top TF-IDF terms:
the 0.3253
and 0.2847
she 0.2643
it 0.2338
was 0.2237
to 0.1932
bats 0.1571
of 0.1423
that 0.122
hall 0.1198
cats 0.1198
eat 0.1145
key 0.1124
in 0.1118
you 0.1118


### B3. Cosine similarity (interpretation)
Using TF‑IDF vectors, I compute cosine similarity between documents. Higher similarity means the documents share more distinctive vocabulary.


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

sim = cosine_similarity(X_tfidf)
np.fill_diagonal(sim, 0)

i, j = np.unravel_index(sim.argmax(), sim.shape)
print("Most similar pair:", (i, j))
print("Similarity score:", sim[i, j])

small = pd.DataFrame(sim[:5, :5], columns=[f"D{k}" for k in range(5)], index=[f"D{k}" for k in range(5)])
small


Most similar pair: (np.int64(26), np.int64(27))
Similarity score: 0.7664234519997631


,D0,D1,D2,D3,D4
D0,0.000000,0.632585,0.650152,0.598930,0.605292
D1,0.632585,0.000000,0.660062,0.612091,0.629080
D2,0.650152,0.660062,0.000000,0.605890,0.664636
D3,0.598930,0.612091,0.605890,0.000000,0.596543
D4,0.605292,0.629080,0.664636,0.596543,0.000000


**Note:** If you observe a *surprising* high similarity between two chunks, it is usually because they share repeated named entities (e.g., character names) or repeated dialogue patterns in that region of the story.


## Part C — Word Embeddings (25%)


In [9]:
from nltk.tokenize import sent_tokenize

raw_sents = sent_tokenize(raw_text)
sentences_tokens = [preprocess(s) for s in raw_sents]
sentences_tokens = [s for s in sentences_tokens if len(s) >= 2]

print("Number of sentences:", len(sentences_tokens))
print("Example sentence:", sentences_tokens[0][:20])


Number of sentences: 1592
Example sentence: ['alice', "'s", 'adventures', 'in', 'wonderland', 'by', 'lewis', 'carroll', '1865', 'chapter', 'i']


## (Required by assignment description) N‑gram Statistical Language Model
This section trains a trigram language model using `nltk.lm` and reports perplexity and sample generations.


In [10]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import Laplace
from nltk.util import bigrams, trigrams
import random

# Use the same preprocessed sentences for LM training
# Split into train/test (90/10) at sentence level
random.seed(42)
sents = sentences_tokens.copy()
random.shuffle(sents)
split = int(0.9 * len(sents))
train_sents = sents[:split]
test_sents = sents[split:]

n = 3  # trigram LM
train_data, padded_vocab = padded_everygram_pipeline(n, train_sents)

lm = Laplace(n)  # Laplace smoothing helps on smaller corpora
lm.fit(train_data, padded_vocab)

# Evaluate perplexity on held-out test set
test_data, _ = padded_everygram_pipeline(n, test_sents)
pp = lm.perplexity([ng for sent in test_data for ng in sent])
print("Trigram LM (Laplace) perplexity on test set:", round(pp, 2))

# Generate a few sample sentences
def generate_sentence(model, num_words=20):
    words = []
    context = ["<s>"] * (n - 1)
    for _ in range(num_words):
        w = model.generate(1, text_seed=context)[0]
        if w == "</s>":
            break
        words.append(w)
        context = (context + [w])[-(n - 1):]
    return " ".join(words)

print("\nGenerated samples:")
for _ in range(3):
    print("-", generate_sentence(lm, num_words=25))


Trigram LM (Laplace) perplexity on test set: 547.0

Generated samples:
- h t < b g < y t < l i s s s o m s h g s i ' < t <
- s < c a m b < h r < h y ' ' < s m s i u t < i d s
- i q p ' s o m b t < < n o m s p y w ' q < < s i w


**Interpretation:** Perplexity reflects how well the n‑gram model predicts unseen text (lower is better). Because this corpus is relatively small and literary, generated sentences may be locally grammatical but can drift semantically.


In [11]:
from gensim.models import Word2Vec

w2v = Word2Vec(
    sentences=sentences_tokens,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,      # skip-gram
    epochs=20
)

print("Word2Vec vocab size:", len(w2v.wv))


Word2Vec vocab size: 1089


In [12]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

# Choose 5 frequent *content* words: exclude stopwords, keep alphabetic tokens, and require presence in Word2Vec vocab
content_candidates = []
for w, c in freq.most_common(300):
    if w.isalpha() and (w not in stop_words) and (len(w) > 2) and (w in w2v.wv):
        content_candidates.append(w)
    if len(content_candidates) >= 5:
        break

targets = content_candidates
print("Targets (frequent content words):", targets)

for w in targets:
    print("\nWord:", w)
    sims = w2v.wv.most_similar(w, topn=10)
    for neigh, score in sims:
        print(neigh, round(score, 3))


Targets (frequent content words): ['said', 'alice', 'little', 'one', 'would']

Word: said
replied 0.795
added 0.774
angrily 0.773
interrupted 0.762
butter 0.749
sharply 0.745
rather 0.741
gravely 0.736
important 0.733
'when 0.731

Word: alice
sharply 0.757
feeling 0.754
'very 0.74
certainly 0.735
'not 0.732
rather 0.727
remarks 0.718
'to 0.716
'for 0.715
very 0.713

Word: little
pattering 0.796
sharp 0.771
box 0.76
tiny 0.76
glass 0.754
inches 0.745
grunted 0.742
crash 0.742
remembered 0.738
golden 0.737

Word: one
eye 0.763
getting 0.736
write 0.729
cake 0.728
corner 0.725
slate 0.719
trying 0.709
full 0.709
arches 0.707
room 0.707

Word: would
should 0.693
beheaded 0.671
hungry 0.662
easily 0.66
happen 0.659
wondering 0.657
otherwise 0.655
whether 0.651
feel 0.644
venture 0.64


### C3. Similarity interpretation
For each target word, the neighbors are words that appear in similar contexts in the story. For example, character names tend to be close to pronouns and verbs used around them, while setting-related words cluster together.


In [13]:
# C4: Analogy queries (vector arithmetic)
# Try a list of candidate analogies and keep the first 3 that are fully in-vocabulary.
candidates = [
    ("alice", "girl", "boy"),
    ("rabbit", "time", "watch"),
    ("queen", "woman", "man"),
    ("king", "man", "woman"),
    ("head", "eyes", "nose"),
    ("cat", "grin", "smile"),
    ("tea", "drink", "eat"),
    ("dormouse", "mouse", "cat"),
]

valid = []
for a, b, c in candidates:
    if a in w2v.wv and b in w2v.wv and c in w2v.wv:
        valid.append((a, b, c))
    if len(valid) >= 3:
        break

print("Analogy queries used:", valid)

for a, b, c in valid:
    print(f"\n{a} - {b} + {c} ≈")
    print(w2v.wv.most_similar(positive=[a, c], negative=[b], topn=5))

if len(valid) < 3:
    print("\nNote: Fewer than 3 analogies were possible because some words are out-of-vocabulary (filtered by min_count) in this small corpus.")


Analogy queries used: [('alice', 'girl', 'boy'), ('rabbit', 'time', 'watch'), ('head', 'eyes', 'nose')]

alice - girl + boy ≈
[('hurried', 0.8002070784568787), ('turning', 0.7772132158279419), ('tossing', 0.7729911208152771), ('impatiently', 0.76960289478302), ('trembling', 0.7674054503440857)]

rabbit - time + watch ≈
[('timid', 0.7417389750480652), ('trembling', 0.7219347953796387), ('voice', 0.7199951410293579), ('loud', 0.7124664783477783), ('trumpet', 0.70660001039505)]

head - eyes + nose ≈
[('tongue', 0.7763422727584839), ('shoes', 0.7663883566856384), ("'hold", 0.7589172720909119), ("'are", 0.7252373099327087), ('whiskers', 0.7185142636299133)]
